# 01 — Chargement de la base FINESS

Charge en parallèle les **EG** (établissements géographiques) et les **EJ** (entités juridiques) depuis SQL Server, en filtrant sur les structures actives.

Sortie : 2 fichiers parquet dans `data/interim/`.

In [2]:
# Installation des packages
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.connexion    import get_finess_connection
from src.display      import afficher_tableau
from config.settings  import (
    FINESS_EG_RAW, FINESS_EJ_RAW, INTERIM_DIR,
)

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 1. EG (établissements géographiques)

In [3]:
conn = get_finess_connection()

query_eg = """
    SELECT idstructure_stru, nmfinessej_stru, nmfinessetab_stru,
           categetab_stru, nmsiret_stru, raisonsociale_stru,
           nmvoie_stru, lbtypevoie_stru, lbvoie_stru, cdcommune_stru
    FROM BICOEUR_DWH_SNAPSHOT.dbo.dwh_structure
    WHERE topsource_stru = 'FINESS'
      AND typeidpm_stru  = 'EG'
      AND (dtfermestruct_stru IS NULL OR dtfermestruct_stru >= SYSDATETIME())
"""
df_eg = pd.read_sql(query_eg, conn)
print(f'EG FINESS actifs : {len(df_eg):,}')
print(f'  avec SIRET     : {df_eg["nmsiret_stru"].notna().sum():,}')

df_eg.to_parquet(FINESS_EG_RAW, index=False)
print(f'Sauvegardé : {FINESS_EG_RAW}')

afficher_tableau(df_eg, 'Aperçu EG FINESS')

/tmp/ipykernel_168/3453939771.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_eg = pd.read_sql(query_eg, conn)


EG FINESS actifs : 104,805
  avec SIRET     : 91,681
Sauvegardé : /home/jovyan/work/projet_finess_sirene/data/interim/finess_eg.parquet


idstructure_stru,nmfinessej_stru,nmfinessetab_stru,categetab_stru,nmsiret_stru,raisonsociale_stru,nmvoie_stru,lbtypevoie_stru,lbvoie_stru,cdcommune_stru
2418252,750063240,030008106,300,83829611900038,INST. SUP. RÉÉDUCATION PSYCHOMOTRICE,20,R,FLEURY,03310
2418253,070007935,070007943,603,90919396300016,MAISON DE SANTE DES TROIS RIVIERES,1,PL,DE LA PETITE VITESSE,07201
2418254,070007950,070007968,699,44059019800014,EML CENTRE IMAGERIE MEDICALE TOURNON,50,R,DES ALPES,07324
2418255,070007984,070007992,603,87958053800012,MAISON DE SANTE HTES VALLEES D'ARDECHE,135,R,LA DAME DE VENTADOUR,07156
2418256,080011042,080011059,603,84806123000019,MSP DE BUZANCY,4,R,DE LA PETITE BAR,08089


## 2. EJ (entités juridiques)

In [4]:
query_ej = """
    SELECT idstructure_stru, nmfinessej_stru, nmfinessetab_stru,
           categetab_stru, nmsiren_stru, nmsiret_stru,
           raisonsociale_stru, nmvoie_stru, lbtypevoie_stru, 
           lbvoie_stru, cdcommune_stru, dtouvertstruct_stru, cdape_stru
    FROM BICOEUR_DWH_SNAPSHOT.dbo.dwh_structure
    WHERE topsource_stru = 'FINESS'
      AND typeidpm_stru  = 'EJ'
      AND (dtfermestruct_stru IS NULL OR dtfermestruct_stru >= SYSDATETIME())
"""
df_ej = pd.read_sql(query_ej, conn)
conn.close()

print(f'EJ FINESS actifs : {len(df_ej):,}')
print(f'  avec SIREN     : {df_ej["nmsiren_stru"].notna().sum():,}')
print(f'  avec APE       : {df_ej["cdape_stru"].notna().sum():,}')
print(f'  avec date ouv. : {df_ej["dtouvertstruct_stru"].notna().sum():,}')

df_ej.to_parquet(FINESS_EJ_RAW, index=False)
print(f'Sauvegardé : {FINESS_EJ_RAW}')

afficher_tableau(df_ej, 'Aperçu EJ FINESS')

/tmp/ipykernel_168/4229738088.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ej = pd.read_sql(query_ej, conn)


EJ FINESS actifs : 54,185
  avec SIREN     : 52,085
  avec APE       : 20,394
  avec date ouv. : 54,163
Sauvegardé : /home/jovyan/work/projet_finess_sirene/data/interim/finess_ej.parquet


idstructure_stru,nmfinessej_stru,nmfinessetab_stru,categetab_stru,nmsiren_stru,nmsiret_stru,raisonsociale_stru,nmvoie_stru,lbtypevoie_stru,lbvoie_stru,cdcommune_stru,dtouvertstruct_stru,cdape_stru
1831435,130804008,None,None,333483667,None,HABITAT PLURIEL,11,R,ARMENY,13206,2001-01-01 00:00:00,8899B
1831436,130804016,None,None,782974158,None,ASSOCIATION LE CANA,514,CHE,DE LA MADRAGUE-VILLE,13215,2001-01-01 00:00:00,8559A
1831438,130804032,None,699,334353471,None,ASSOCIATION REGIONALE POUR INTEGRATION,26,R,SAINT SEBASTIEN,13206,2001-01-01 00:00:00,8891B
1831440,130804057,None,699,775559701,None,ENTRAIDE,13,R,ROUX DE BRIGNOLES,13206,2001-01-01 00:00:00,8810A
1831441,130804065,None,None,775558364,None,CAF 13,215,CHE,DE GIBBES,13214,2001-01-01 00:00:00,8430C
